In [ ]:
#0 Load Libraries and Configurations

import os
import wrds
import pandas as pd

# ---------- User-configurable paths ----------
path_data_intermediate = "/path/to/intermediate"  # <-- set this
os.makedirs(path_data_intermediate, exist_ok=True)

parquet_out = os.path.join(path_data_intermediate, "CompustatBSegments.parquet")
csv_out     = os.path.join(path_data_intermediate, "CompustatBSegments.csv")

In [ ]:
#1 Load Compustat Data

SQL = """
SELECT
    a.gvkey,
    a.datadate,
    a.stype,
    a.sid,
    a.sales,
    a.srcdate,
    a.naicsh,
    a.sics1,
    a.snms
FROM compseg.wrds_segmerged AS a
WHERE a.datadate >= DATE '2000-01-01';
"""

In [ ]:
#2 Compustat Data Extraction From WRDS

db = wrds.Connection()  # prompts for creds if not cached
df = db.raw_sql(SQL, date_cols=["datadate", "srcdate"])

In [ ]:
#3 Data Cleaning

# ------------ Destring (gvkey, sics1, naicsh) ------------
# Convert to numeric where possible; otherwise leaves as original (string) if non-numeric.
for col in ["gvkey", "sics1", "naicsh"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="ignore")

# Optional: sort for readability
df = df.sort_values(["gvkey", "datadate", "sid"], kind="mergesort")

# ------------ Save outputs ------------
df.to_parquet(parquet_out, index=False)
df.to_csv(csv_out, index=False)

print("Saved:")
print(" -", parquet_out)
print(" -", csv_out)
print(df.head())